# 14 — Version 2 tabular neural network

**Owners:** Mirdula / Hashvitha

Version 1 was still improving at its final allowed epoch. Version 2 makes the
recorded architecture genuinely configurable, permits up to 50 epochs, uses a
learning-rate scheduler, and compares full versus square-root class weighting.


## What Version 2 changes—and why

Version 1 remains our reproducible baseline. Version 2 adds behaviour that the
winning Kaggle solution showed was valuable, but implements it in a stricter
real-time form:

- `D` values are normalized against transaction day to expose stable date anchors.
- a conservative `uid_proxy` describes a possible client without using it as a label;
- counts, time since previous use, amount history and unique-value history describe
  behaviour;
- every historical feature uses only earlier transactions; and
- no feature reads `isFraud`, later rows, validation labels, or test labels.

The newest 15% remains the final test period. It is never used for feature or
hyperparameter selection.


In [ ]:
from pathlib import Path
_start = Path.cwd().resolve()
for _candidate in [_start, *_start.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

V2_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "v2"
V2_ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v2"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Project root:", PROJECT_ROOT)
print("Version 2 data:", V2_DATA_DIR)
print("Version 2 artifacts:", V2_ARTIFACT_ROOT)


In [ ]:
required = [V2_DATA_DIR / name for name in ["train.parquet", "validation.parquet", "test.parquet"]]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run 10_v2_behavioral_data_preparation.ipynb first. Missing: " + ", ".join(missing)
    )

train, validation, test = [pd.read_parquet(path) for path in required]
FAST_RUN = False  # Only for code checks. Never report FAST_RUN metrics.
if FAST_RUN:
    def debug_sample(frame, rows):
        return (frame.groupby("isFraud", group_keys=False)
                .apply(lambda group: group.sample(
                    n=max(1, round(rows * len(group) / len(frame))),
                    random_state=RANDOM_SEED), include_groups=True)
                .sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True))
    train = debug_sample(train, 60_000)
    validation = debug_sample(validation, 20_000)
    test = debug_sample(test, 20_000)

TARGET = "isFraud"
DROP_FROM_MODEL = ["isFraud", "TransactionID"]
X_train, y_train = train.drop(columns=DROP_FROM_MODEL), train[TARGET].astype("int8")
X_validation, y_validation = validation.drop(columns=DROP_FROM_MODEL), validation[TARGET].astype("int8")
X_test, y_test = test.drop(columns=DROP_FROM_MODEL), test[TARGET].astype("int8")
development = pd.concat([train, validation], ignore_index=True)
print("Train:", X_train.shape, "fraud rate:", f"{y_train.mean():.4%}")
print("Validation:", X_validation.shape, "fraud rate:", f"{y_validation.mean():.4%}")
print("Test:", X_test.shape, "fraud rate:", f"{y_test.mean():.4%}")


In [ ]:
MODEL_KEY = "neural_network"

from datetime import datetime, timezone
from src.fraud_pipeline.artifacts import build_manifest, package_versions, write_json
from src.fraud_pipeline.evaluation import evaluate_binary_classifier, select_operating_threshold

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = V2_ARTIFACT_ROOT / MODEL_KEY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("This Version 2 run will be saved to:", RUN_DIR)


In [ ]:
import copy, joblib, torch
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from src.fraud_pipeline.neural_v2 import (
    FraudTabularNetworkV2, embedding_dimension_v2, network_v2_from_config,
)
from src.fraud_pipeline.preprocessing import NeuralTabularPreprocessor
from src.fraud_pipeline.validation_v2 import positive_weight

torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
preprocessor = NeuralTabularPreprocessor(rare_min_count=20).fit(X_train)
train_numeric, train_categorical = preprocessor.transform(X_train)
validation_numeric, validation_categorical = preprocessor.transform(X_validation)
embedding_dimensions = [embedding_dimension_v2(value) for value in preprocessor.cardinalities]
model_config = {
    "numeric_size": preprocessor.numeric_output_size,
    "cardinalities": preprocessor.cardinalities,
    "embedding_dimensions": embedding_dimensions,
    "categorical_columns": preprocessor.categorical_columns,
    "hidden_layers": [384, 192, 96], "dropout": [0.30, 0.20, 0.10],
}
print("Device:", device)
print("Numeric inputs:", model_config["numeric_size"])
print("Categorical inputs:", len(model_config["cardinalities"]))


## Data loaders and reusable training function


In [ ]:
BATCH_SIZE = 4096
MAX_EPOCHS = 50
PATIENCE = 7
train_dataset = TensorDataset(
    torch.from_numpy(train_numeric), torch.from_numpy(train_categorical),
    torch.from_numpy(y_train.to_numpy(dtype=np.float32)))
validation_dataset = TensorDataset(
    torch.from_numpy(validation_numeric), torch.from_numpy(validation_categorical),
    torch.from_numpy(y_validation.to_numpy(dtype=np.float32)))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=device.type == "cuda")
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE * 2,
    shuffle=False, num_workers=2, pin_memory=device.type == "cuda")

def predict_loader(model, loader):
    model.eval(); probabilities, labels = [], []
    with torch.inference_mode():
        for numeric, categorical, target in loader:
            logits = model(numeric.to(device), categorical.to(device))
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(target.numpy())
    return np.concatenate(probabilities), np.concatenate(labels)

def train_candidate(weight_mode):
    torch.manual_seed(RANDOM_SEED)
    model = network_v2_from_config(model_config).to(device)
    weight = positive_weight(y_train, weight_mode)
    loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    best_score, best_state, without_improvement, history = -np.inf, None, 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train(); total_loss = 0.0
        for numeric, categorical, target in train_loader:
            numeric = numeric.to(device, non_blocking=True)
            categorical = categorical.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=device.type == "cuda"):
                loss = loss_function(model(numeric, categorical), target)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * len(target)
        probability, label = predict_loader(model, validation_loader)
        score = average_precision_score(label, probability)
        scheduler.step(score)
        row = {"weight_mode": weight_mode, "epoch": epoch,
               "train_loss": total_loss / len(train_dataset),
               "validation_pr_auc": score,
               "learning_rate": optimizer.param_groups[0]["lr"]}
        history.append(row); print(row)
        if score > best_score + 1e-5:
            best_score = score
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            without_improvement = 0
        else:
            without_improvement += 1
            if without_improvement >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best_score, weight, history


## Compare class-weight strength on validation only


In [ ]:
candidate_modes = ["sqrt_balanced", "balanced"] if not FAST_RUN else ["sqrt_balanced"]
candidate_results = []
started = time.perf_counter()
for mode in candidate_modes:
    candidate_model, score, weight, history = train_candidate(mode)
    candidate_results.append({"mode": mode, "score": score, "weight": weight,
                              "state": copy.deepcopy(candidate_model.state_dict()),
                              "history": history})
    del candidate_model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
best = max(candidate_results, key=lambda item: item["score"])
model = network_v2_from_config(model_config).to(device)
model.load_state_dict(best["state"])
training_seconds = time.perf_counter() - started
history = pd.DataFrame([row for item in candidate_results for row in item["history"]])
display(history.groupby("weight_mode")["validation_pr_auc"].max())
print("Selected:", best["mode"], "PR-AUC:", best["score"])


## Final validation and untouched test


In [ ]:
validation_probability, _ = predict_loader(model, validation_loader)
threshold_record = select_operating_threshold(y_validation, validation_probability, minimum_precision=0.10)
threshold = float(threshold_record["threshold"])
validation_metrics = evaluate_binary_classifier(y_validation, validation_probability, threshold)
test_numeric, test_categorical = preprocessor.transform(X_test)
test_dataset = TensorDataset(torch.from_numpy(test_numeric), torch.from_numpy(test_categorical),
                             torch.from_numpy(y_test.to_numpy(dtype=np.float32)))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                         num_workers=2, pin_memory=device.type == "cuda")
test_probability, _ = predict_loader(model, test_loader)
test_metrics = evaluate_binary_classifier(y_test, test_probability, threshold)
display(pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])[
    ["pr_auc", "roc_auc", "precision", "recall", "f1", "brier_score"]])


## Save, reload, and package


In [ ]:
torch.save({"model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "model_config": model_config, "best_validation_pr_auc": best["score"]}, RUN_DIR / "model.pt")
joblib.dump(preprocessor, RUN_DIR / "numeric_and_categorical_preprocessor.joblib", compress=3)
history.to_csv(RUN_DIR / "training_history.csv", index=False)
write_json(RUN_DIR / "model_config.json", model_config)
pd.DataFrame({"TransactionID": validation.TransactionID, "isFraud": y_validation,
              "probability": validation_probability}).to_parquet(RUN_DIR / "validation_predictions.parquet", index=False)
pd.DataFrame({"TransactionID": test.TransactionID, "isFraud": y_test,
              "probability": test_probability}).to_parquet(RUN_DIR / "test_predictions.parquet", index=False)
write_json(RUN_DIR / "threshold.json", threshold_record)
write_json(RUN_DIR / "metrics.json", {"validation": validation_metrics, "test": test_metrics})
write_json(RUN_DIR / "feature_schema.json", {"groups": preprocessor.groups,
           "categorical_columns": preprocessor.categorical_columns,
           "behavioral_contract": "data/processed/v2/behavioral_contract.json"})
write_json(RUN_DIR / "training_config.json", {
    "model": "v2_neural_network", "run_id": RUN_ID, "fast_run": FAST_RUN,
    "random_seed": RANDOM_SEED, "training_seconds": training_seconds,
    "device": str(device), "selected_weight_mode": best["mode"],
    "positive_weight": best["weight"], "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
    "versions": package_versions(["numpy", "pandas", "scikit-learn", "torch", "joblib"]),
})
loaded_preprocessor = joblib.load(RUN_DIR / "numeric_and_categorical_preprocessor.joblib")
checkpoint = torch.load(RUN_DIR / "model.pt", map_location=device, weights_only=True)
loaded_model = network_v2_from_config(checkpoint["model_config"]).to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
sample_numeric, sample_categorical = loaded_preprocessor.transform(X_validation.iloc[:5])
loaded_model.eval()
with torch.inference_mode():
    reloaded = torch.sigmoid(loaded_model(torch.from_numpy(sample_numeric).to(device),
        torch.from_numpy(sample_categorical).to(device))).cpu().numpy()
np.testing.assert_allclose(validation_probability[:5], reloaded, rtol=1e-5, atol=1e-7)


In [ ]:
import shutil
write_json(RUN_DIR / "manifest.json", build_manifest(RUN_DIR))
archive_base = RUN_DIR.parent / f"{MODEL_KEY}_{RUN_ID}"
archive_path = Path(shutil.make_archive(str(archive_base), "gztar", root_dir=RUN_DIR))
print("Reload check passed.")
print("Artifact folder:", RUN_DIR)
print("Share this archive:", archive_path)
